# 02 Feature Experiments
Prototype feature ideas here, then move stable code to `src/features/`.

In [ ]:
from pathlib import Path
print('Notebook ready for feature experiments')
print(Path.cwd())

Notebook ready for feature experiments
c:\temp\ml_demo1\notebooks


# 02 — Feature Engineering: From Notebook to Repo

**Goal:** Turn EDA findings into repeatable, engineered features.

The lesson arc:
1. **Why raw features break models** — demonstrate with a measurable experiment
2. **Build each transform by hand** — imputation, scaling, encoding
3. **The leakage trap** — the most common beginner mistake
4. **Productize** — call `src/features/preprocess.py` directly
5. **Configure** — change behaviour via `configs/features/default.yaml`
6. **Full flow diagram** — trace notebook → repo → serialized model

> **Rule:** Prototype here, then move stable logic to `src/`. Never keep preprocessing only in a notebook.

In [31]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore')

_cwd = Path().resolve()
REPO_ROOT = next((p for p in [_cwd, *_cwd.parents] if (p / 'requirements.txt').exists()), _cwd)
sys.path.insert(0, str(REPO_ROOT))

df = pd.read_csv(REPO_ROOT / 'data' / 'processed' / 'train.csv')

TARGET = 'target'
X = df.drop(columns=[TARGET])
y = df[TARGET]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset : {X.shape[0]} rows, {X.shape[1]} features")
print(f"Train   : {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

Dataset : 569 rows, 30 features
Train   : 455 rows | Test: 114 rows


In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Model A: raw features (no scaling)
try:
    m_raw = LogisticRegression(max_iter=200, random_state=42)
    m_raw.fit(X_train, y_train)
    auc_raw = roc_auc_score(y_test, m_raw.predict_proba(X_test)[:, 1])
except Exception as e:
    auc_raw = float('nan')
    print(f"Raw model issue: {e}")

# Model B: scaled features (fit scaler on train only)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)          # <-- transform only, NOT fit
m_sc = LogisticRegression(max_iter=1000, random_state=42)
m_sc.fit(X_tr_sc, y_train)
auc_scaled = roc_auc_score(y_test, m_sc.predict_proba(X_te_sc)[:, 1])

print(f"ROC-AUC raw features  : {auc_raw:.4f}")
print(f"ROC-AUC scaled features: {auc_scaled:.4f}")
print("\nLesson: scaling matters even for logistic regression.")

ROC-AUC raw features  : 0.9957
ROC-AUC scaled features: 0.9954

Lesson: scaling matters even for logistic regression.


## Part 2 — Build Each Transform by Hand

Understanding each step makes `src/features/preprocess.py` readable rather than a black box.

### Step 2a — Imputation

In [33]:
from sklearn.impute import SimpleImputer

# Inject 5% artificial missing values to demonstrate imputation
rng = np.random.default_rng(42)
X_gaps = X_train.copy()
mask = rng.random(X_gaps.shape) < 0.05
X_gaps[mask] = np.nan

print(f"Missing before imputation: {X_gaps.isnull().sum().sum()}")

imputer = SimpleImputer(strategy='median')   # fit on train only
X_imputed = pd.DataFrame(imputer.fit_transform(X_gaps), columns=X_gaps.columns)

print(f"Missing after imputation : {X_imputed.isnull().sum().sum()}")
print("\nImputed medians (first 5 columns):")
print(pd.Series(imputer.statistics_[:5], index=X_gaps.columns[:5]).round(4))

Missing before imputation: 671
Missing after imputation : 0

Imputed medians (first 5 columns):
mean radius         13.2250
mean texture        18.7600
mean perimeter      86.3400
mean area          540.3500
mean smoothness      0.0957
dtype: float64


### Step 2b — Scaling

In [35]:
sc = StandardScaler()
X_tr_sc2 = pd.DataFrame(sc.fit_transform(X_train), columns=X_train.columns)

print("Before StandardScaler — first 3 columns (mean / std):")
print(X_train.iloc[:, :3].agg(['mean', 'std']).round(3))

print("\nAfter StandardScaler — first 3 columns (mean / std):")
print(X_tr_sc2.iloc[:, :3].agg(['mean', 'std']).round(3))
print("\nMean ≈ 0, Std ≈ 1 after scaling.")

Before StandardScaler — first 3 columns (mean / std):
      mean radius  mean texture  mean perimeter
mean       14.067        19.247          91.557
std         3.499         4.405          24.149

After StandardScaler — first 3 columns (mean / std):
      mean radius  mean texture  mean perimeter
mean       -0.000         0.000          -0.000
std         1.001         1.001           1.001

Mean ≈ 0, Std ≈ 1 after scaling.


## Part 3 — The Leakage Trap ⚠️

**The most common mistake in ML preprocessing.**

If you fit the scaler on the *full dataset* before splitting, test statistics leak into training. The model appears better than it really is.

**Rule: fit only on `X_train`, then `transform` on `X_test`.**

The sklearn `Pipeline` enforces this automatically — which is exactly why `src/features/preprocess.py` wraps everything in a `Pipeline`.

In [36]:
# BAD — fit on full dataset BEFORE split (leaks test info into training)
bad_scaler = StandardScaler()
X_all_scaled = bad_scaler.fit_transform(X)           # <-- WRONG
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_all_scaled, y.values, test_size=0.2, random_state=42, stratify=y
)
m_bad = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_bad, y_tr_bad)
auc_bad = roc_auc_score(y_te_bad, m_bad.predict_proba(X_te_bad)[:, 1])

# GOOD — fit on train only, transform test
good_scaler = StandardScaler()
X_tr_good = good_scaler.fit_transform(X_train)       # <-- fit on train
X_te_good = good_scaler.transform(X_test)            # <-- transform only
m_good = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr_good, y_train)
auc_good = roc_auc_score(y_test, m_good.predict_proba(X_te_good)[:, 1])

print(f"BAD  (leaky scaler)  ROC-AUC: {auc_bad:.4f}  <-- inflated")
print(f"GOOD (safe scaler)   ROC-AUC: {auc_good:.4f}  <-- trustworthy")
print()
print("The sklearn Pipeline in src/features/preprocess.py enforces the GOOD approach.")

BAD  (leaky scaler)  ROC-AUC: 0.9954  <-- inflated
GOOD (safe scaler)   ROC-AUC: 0.9954  <-- trustworthy

The sklearn Pipeline in src/features/preprocess.py enforces the GOOD approach.


## Part 4 — The Production Pipeline in `src/`

The manual steps above are consolidated into `src/features/preprocess.py`. Let's call it directly and inspect what it builds.

In [37]:
from src.features.preprocess import build_preprocessor

preprocessor = build_preprocessor(X_train, scale_numeric=True)

# Inspect which columns go to which branch
for name, pipe, cols in preprocessor.transformers:
    steps = [s[0] for s in pipe.steps]
    print(f"Branch '{name}': {len(cols)} column(s) → steps: {steps}")

Branch 'num': 30 column(s) → steps: ['imputer', 'scaler']
Branch 'cat': 0 column(s) → steps: ['imputer', 'onehot']


In [38]:
from sklearn.pipeline import Pipeline as SKPipeline

# Combine preprocessor + model — same as train.py does internally
full_pipeline = SKPipeline([
    ('preprocessor', build_preprocessor(X_train, scale_numeric=True)),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

full_pipeline.fit(X_train, y_train)
auc = roc_auc_score(y_test, full_pipeline.predict_proba(X_test)[:, 1])

print(f"Pipeline ROC-AUC: {auc:.4f}")
print()
print("This pipeline object is what gets saved to runs/run_001/model.joblib")
print("Preprocessing is bundled inside — predict.py never re-implements transforms.")

Pipeline ROC-AUC: 0.9954

This pipeline object is what gets saved to runs/run_001/model.joblib
Preprocessing is bundled inside — predict.py never re-implements transforms.


## Part 5 — Config Controls Feature Behaviour

`configs/features/default.yaml` controls what `build_preprocessor` does. Students should change a value, re-run `train.py`, and compare `runs/summary.csv`.

In [39]:
import yaml

cfg_path = REPO_ROOT / 'configs' / 'features' / 'default.yaml'
with open(cfg_path) as f:
    feature_cfg = yaml.safe_load(f)

print("configs/features/default.yaml:")
for k, v in feature_cfg.items():
    print(f"  {k}: {v}")

print()
print("Exercise: set scale_numeric to false, change run_name to run_002 in")
print("configs/train/default.yaml, re-run train.py, then compare runs/summary.csv.")

configs/features/default.yaml:
  drop_columns: []
  scale_numeric: True
  one_hot_encode: True

Exercise: set scale_numeric to false, change run_name to run_002 in
configs/train/default.yaml, re-run train.py, then compare runs/summary.csv.


## Part 6 — Experiment: Drop Highly Correlated Features

From EDA we found feature pairs with |correlation| > 0.9. Let's test whether dropping them changes model quality — and if not, they are redundant and safe to remove.

In [40]:
def run_experiment(X_tr, X_te, label):
    pre = build_preprocessor(X_tr, scale_numeric=True)
    pipe = SKPipeline([('pre', pre), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
    pipe.fit(X_tr, y_train)
    auc = roc_auc_score(y_test, pipe.predict_proba(X_te)[:, 1])
    print(f"{label:<45} ROC-AUC = {auc:.4f}")

# Baseline
run_experiment(X_train, X_test, "All features")

# Drop one member of a highly correlated pair found in EDA
# (students should substitute columns from their own EDA heatmap)
drop_cols = ['perimeter_mean']
run_experiment(
    X_train.drop(columns=drop_cols, errors='ignore'),
    X_test.drop(columns=drop_cols, errors='ignore'),
    f"Drop {drop_cols}"
)

print()
print("If AUC is unchanged -> feature is redundant.")
print("If AUC drops       -> feature carries unique signal, keep it.")
print()
print("Confirmed drops go into configs/features/default.yaml -> drop_columns: [...]")

All features                                  ROC-AUC = 0.9954
Drop ['perimeter_mean']                       ROC-AUC = 0.9954

If AUC is unchanged -> feature is redundant.
If AUC drops       -> feature carries unique signal, keep it.

Confirmed drops go into configs/features/default.yaml -> drop_columns: [...]


## Part 7 — Full Flow: Notebook → Repo → Serialized Model

```
01_eda.ipynb
  findings: different scales, correlated pairs, no missing values
       ↓
02_feature_experiments.ipynb     (this notebook)
  prototype transforms, measure impact of each decision
       ↓
src/features/preprocess.py       reusable build_preprocessor()
configs/features/default.yaml   scale_numeric, drop_columns, one_hot_encode
       ↓
train.py                         reads config, builds pipeline, fits model
       ↓
runs/run_001/model.joblib        Pipeline(preprocessor + model) — transforms bundled in
       ↓
predict.py                       loads model.joblib, calls .predict() — no separate preprocessing
```

**The guarantee:** because the full `Pipeline` (preprocessing + model) is serialized together, inference always uses exactly the same transforms as training. This is enforced by the repo structure, not by convention.